### This Notebook: Continued Pretraining (CPT) on Business Central AL Code

This notebook is configured for **continued pretraining** of Gemma 4 E2B on Microsoft Dynamics 365 Business Central AL source code in Kaggle.

**Key differences from instruction fine-tuning:**
- Uses raw text completion (next-token prediction), not chat format
- Trains on ALL tokens (no masking of user vs assistant parts)
- Includes `embed_tokens` and `lm_head` in LoRA for domain adaptation
- Uses `UNSLOTH_RETURN_LOGITS=1` to disable CCE (not supported for CPT)
- Higher LoRA rank (128) and rank-stabilized LoRA for better adaptation
- EOS tokens added to help the model learn proper stopping

**Data source:** a Kaggle dataset containing `business_central_al_training_text.txt`, generated beforehand from `.al` files with `scripts/convert_al_to_text.py`.

---

### Installation

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

# Set environment variable to disable CCE for continued pretraining.
# CCE (Constant Cross Entropy) is not supported for CPT.
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

!uv pip install -qqq     "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes     datasets unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel
import torch

MODEL_NAME = "unsloth/gemma-4-E2B"
MAX_SEQ_LENGTH = 8192

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
]

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None, # None for auto detection
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
    device_map = {"": torch.cuda.current_device()},
)

In [ ]:
from transformers import TextStreamer

# Helper function for chat inference (for testing before CPT)
def do_gemma_4_chat_inference(messages, max_new_tokens = 128):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0,
        top_p = 0.95,
        top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True),
    )


def _get_text_tokenizer():
    return getattr(tokenizer, "tokenizer", tokenizer)


def _unused_token_ids(text_tokenizer, limit=256):
    ids = []
    for index in range(limit):
        token = f"<unused{index}>"
        token_id = text_tokenizer.convert_tokens_to_ids(token)
        if token_id is not None and token_id != text_tokenizer.unk_token_id:
            ids.append([token_id])
    return ids


# Helper function for continued pretraining inference (text completion)
def do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2):
    """For CPT inference - direct text completion tuned for code-like output."""
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread

    device = "cuda" if torch.cuda.is_available() else "cpu"
    text_tokenizer = _get_text_tokenizer()

    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    streamer = TextIteratorStreamer(text_tokenizer, skip_prompt=True, skip_special_tokens=True)

    do_sample = temperature is not None and temperature > 0
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=0.9 if do_sample else None,
        top_k=40 if do_sample else None,
        repetition_penalty=1.05,
        eos_token_id=text_tokenizer.eos_token_id,
        pad_token_id=text_tokenizer.pad_token_id or text_tokenizer.eos_token_id,
        bad_words_ids=_unused_token_ids(text_tokenizer),
    )
    generation_kwargs = {key: value for key, value in generation_kwargs.items() if value is not None}

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""
    for text in streamer:
        print(text, end="", flush=True)
        generated_text += text
    thread.join()
    print()
    return generated_text


# Let's finetune Gemma 4!

You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Turn off for text CPT
    finetune_language_layers   = True,   # Keep on for continued pretraining
    finetune_attention_modules = True,   # Good for CPT
    finetune_mlp_modules       = True,   # Keep on always

    # Embeddings and lm_head help domain adaptation, but they add trainable params.
    # Keep them enabled for quality; disable only if you need an even faster smoke run.
    finetune_embedding_modules = True,
    finetune_lm_head          = True,

    r = 64,            # Faster Kaggle default. Use 128 for a slower, stronger run.
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,
)

### Data Prep for Business Central Continued Pretraining

For continued pretraining, we load the generated Business Central AL training text and train the model to predict the next token. We don't use chat templates - just raw AL text chunks.

Load `business_central_al_training_text.txt` from an attached Kaggle dataset, tokenize it, and create overlapping chunks for training.

In [ ]:
from pathlib import Path
from datasets import Dataset

max_seq_length = MAX_SEQ_LENGTH

# Kaggle usage:
# 1. Generate business_central_al_training_text.txt locally with scripts/convert_al_to_text.py.
# 2. Upload business_central_al_training_text.txt as a Kaggle Dataset.
# 3. Attach that Dataset to this notebook.
# 4. Optionally set KAGGLE_TRAINING_TEXT_PATH to the exact attached dataset path.
KAGGLE_TRAINING_TEXT_PATH = None  # Example: "/kaggle/input/bc-al-training-text/business_central_al_training_text.txt"

if KAGGLE_TRAINING_TEXT_PATH:
    training_text_path = Path(KAGGLE_TRAINING_TEXT_PATH)
else:
    training_text_candidates = sorted(Path("/kaggle/input").glob("**/business_central_al_training_text.txt"))
    if not training_text_candidates:
        training_text_candidates = sorted(Path("/kaggle/input").glob("**/corpus.txt")) # Legacy fallback.
    training_text_path = training_text_candidates[0] if training_text_candidates else None

if training_text_path is None or not training_text_path.exists():
    raise FileNotFoundError(
        "Could not find business_central_al_training_text.txt in an attached Kaggle dataset. "
        "Upload the generated training text as a Kaggle Dataset and attach it to this notebook. "
        "Expected a file like /kaggle/input/<dataset-slug>/business_central_al_training_text.txt."
    )

raw_text = training_text_path.read_text(encoding="utf-8")
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

token_ids = backend_tokenizer.encode(raw_text, add_special_tokens=False)
CHUNK_TOKENS = min(4096, max_seq_length - 1)
OVERLAP_TOKENS = 128
MIN_CHUNK_TOKENS = 256
step = CHUNK_TOKENS - OVERLAP_TOKENS

chunks = []
for start in range(0, len(token_ids), step):
    chunk_ids = token_ids[start:start + CHUNK_TOKENS]
    if len(chunk_ids) < MIN_CHUNK_TOKENS and chunks:
        break
    chunks.append(backend_tokenizer.decode(chunk_ids))

dataset = Dataset.from_dict({"text": chunks})

print(f"Training text path: {training_text_path}")
print(f"Training text characters: {len(raw_text):,}")
print(f"Training text tokens: {len(token_ids):,}")
print(f"Chunk tokens: {CHUNK_TOKENS:,}")
print(f"Overlap tokens: {OVERLAP_TOKENS:,}")
print(f"Dataset size: {len(dataset):,} examples")

Let's verify the data by looking at a sample chunk:

In [ ]:
print(f"Sample chunk (first 500 chars):\n{'='*50}\n")
print(dataset[0]["text"][:500])
print(f"\n{'='*50}\n")
print(f"Sample chunk (middle of training text, first 500 chars):\n{'='*50}\n")
print(dataset[len(dataset)//2]["text"][:500])

For continued pretraining, we use the raw text directly without chat templates. The model will learn to predict the next token.

In [ ]:
# For continued pretraining, we just use the text as-is
# No chat template formatting needed - this is pure next-token prediction

# Let's see the first example
dataset[0]["text"][:200]

The AL text is already in the correct format for continued pretraining - we pass raw code text to the model and it learns to predict the next token.

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def add_eos_token(examples):
    """Add EOS token to each text chunk for proper next-token prediction."""
    texts = examples["text"]
    return {"text": [text + EOS_TOKEN for text in texts]}

# Apply EOS token addition before splitting.
dataset = dataset.map(add_eos_token, batched=True)

# Hold out a small validation split so loss tells us whether the model is learning
# general AL patterns instead of only memorizing the training chunks.
split_dataset = dataset.train_test_split(test_size=0.02, seed=3407, shuffle=True)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Dataset features: {dataset.features}")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(eval_dataset):,}")
print(f"EOS token: {repr(EOS_TOKEN)}")

The dataset is ready. Each example contains a chunk of text that will be used for next-token prediction training.

In [ ]:
# Look at an example
print(train_dataset[0]["text"][:300] + "...")

<a name="BaselineInference"></a>
### Baseline AL Completions Before Training

Run the fixed AL prompts before training so you can compare the base model against the continued-pretrained adapter later.


In [ ]:
# Fixed AL prompts for checking Business Central code completion quality.
test_prompts = [
    r"""tableextension 50100 CustomerExt extends Customer
{
    fields
    {
        field(50100; "Credit Risk Score"; Integer)
        {
            Caption = 'Credit Risk Score';""",
    r"""codeunit 50100 SalesOrderValidation
{
    procedure ValidateCustomerCreditLimit(SalesHeader: Record "Sales Header")
    var
        Customer: Record Customer;
    begin""",
    r"""pageextension 50100 CustomerCardExt extends "Customer Card"
{
    layout
    {
        addlast(General)
        {
            field("Credit Risk Score"; Rec."Credit Risk Score")""",
    r"""report 50100 "Customer Balance Export"
{
    UsageCategory = ReportsAndAnalysis;
    ApplicationArea = All;
    dataset
    {
        dataitem(Customer; Customer)""",
    r"""enum 50100 "BC Integration Status"
{
    Extensible = true;

    value(0; Pending)
    {
        Caption = 'Pending';""",
    r"""codeunit 50101 ItemAvailabilityMgt
{
    procedure GetAvailableInventory(ItemNo: Code[20]; LocationCode: Code[10]): Decimal
    var
        Item: Record Item;
    begin""",
]

FastModel.for_inference(model)

print("BASELINE COMPLETIONS BEFORE TRAINING")
print("=" * 50)
for prompt in test_prompts:
    print(f"\nPrompt:\n{prompt}")
    print("-" * 50)
    do_gemma_4_text_completion(prompt, max_new_tokens=192, temperature=0.2)

if hasattr(FastModel, "for_training"):
    FastModel.for_training(model)
else:
    model.train()


<a name="Train"></a>
### Train the model for Business Central Continued Pretraining

For continued pretraining, we train on all tokens. The goal is to adapt Gemma 4 E2B to AL syntax, Business Central object patterns, and Base Application coding conventions.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Kaggle sessions are time-limited. Use a fractional epoch so runtime scales
# with the actual dataset size instead of a fixed step count.
NUM_TRAIN_EPOCHS = 0.5
EVAL_AND_SAVE_STEPS = 350

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = CHUNK_TOKENS + 1,
        packing = False,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.03,
        num_train_epochs = NUM_TRAIN_EPOCHS,
        learning_rate = 1e-5,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = EVAL_AND_SAVE_STEPS,
        save_strategy = "steps",
        save_steps = EVAL_AND_SAVE_STEPS,
        save_total_limit = 2,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

For continued pretraining, we train on ALL tokens (not just responses). We skip the `train_on_responses_only` step since we want the model to learn the full text distribution.

Let's verify the training setup by checking a tokenized example:

In [ ]:
# Tokenize and check the first training example
sample_text = train_dataset[0]["text"]
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
input_ids = backend_tokenizer.encode(sample_text, truncation=True, max_length=max_seq_length)
decoded = backend_tokenizer.decode(input_ids)

print(f"Original length: {len(sample_text)} chars")
print(f"Tokenized length: {len(input_ids)} tokens")
print(f"Trainer max_length: {CHUNK_TOKENS + 1} tokens")
print(f"\nFirst 200 chars of decoded:\n{decoded[:200]}...")

Verify that labels match the input_ids (next-token prediction):

In [ ]:
# Check that we're training on all tokens.
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
input_ids = backend_tokenizer.encode(train_dataset[0]["text"], truncation=True, max_length=max_seq_length)
print("Input IDs (first 10):", input_ids[:10])
print("Labels are the same sequence shifted internally for next-token prediction.")
print(f"\nFor continued pretraining, all {len(input_ids)} non-padding tokens in this chunk are used for training.")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Let's train the model!

This Kaggle-friendly default uses `NUM_TRAIN_EPOCHS = 0.5`, so the run trains on half of one pass through the training chunks. If it finishes comfortably and validation loss is still improving, increase it to `0.75` or `1.0` in a later run.

To resume a training run from a saved checkpoint, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference for Business Central Continued Pretraining

For text completion inference after continued pretraining, use the model as a code completion model. The prompts below start AL objects or procedures and let the model continue them.

In [ ]:
# Prepare for post-training inference.
FastModel.for_inference(model)

print("POST-TRAINING COMPLETION")
print("=" * 50)
prompt = test_prompts[0]
print(f"Prompt:\n{prompt}")
print("-" * 50)
do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2)

### Try more prompts

Test the model with different AL prompts to check whether it learned object structure, triggers, procedures, and Business Central naming patterns.

In [ ]:
# Test with different prompts.
for prompt in test_prompts:
    print(f"\nPrompt:\n{prompt}")
    print("-" * 50)
    do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-4-e2b-bc-cpt")
tokenizer.save_pretrained("gemma-4-e2b-bc-cpt")
# model.push_to_hub("HF_ACCOUNT/gemma-4-e2b-bc-cpt", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-4-e2b-bc-cpt", token = "YOUR_HF_TOKEN")

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma-4-e2b-bc-cpt",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = True,
    )

FastModel.for_inference(model)
do_gemma_4_text_completion(
    r"""codeunit 50100 CustomerPostingHelper
{
    procedure CheckPostingGroup(Customer: Record Customer)
    begin""",
    max_new_tokens = 512,
    temperature = 0.2,
)

### Saving to float16 for VLLM

You can also save the merged model for deployment. Set `if False` to `if True` when you want to export it.

In [ ]:
if False: # Change to True to save merged finetune.
    model.save_pretrained_merged("gemma-4-e2b-bc-cpt-merged", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload merged finetune.
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-e2b-bc-cpt-merged", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF.
    model.save_pretrained_gguf(
        "gemma-4-e2b-bc-cpt-gguf",
        tokenizer,
        quantization_method = "Q8_0",
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF.
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma-4-e2b-bc-cpt-gguf",
        tokenizer,
        quantization_method = "Q8_0",
        token = "YOUR_HF_TOKEN",
    )